# UCI Online Retail II 数据质量复核
公开历史零售明细；不是人力数据，也不是因果实验。源文件和指标定义见 ../docs/真实零售数据接入.md。以下只读检查可用项目虚拟环境执行；不修改原始数据。保留重复和缺失客户记录。

In [ ]:
from pathlib import Path
import json, duckdb
root = Path.cwd() if (Path.cwd() / 'datasets').exists() else Path.cwd().parent
data = root / 'datasets' / 'uci_online_retail_ii'
db = duckdb.connect(str(data / 'reference.duckdb'), read_only=True)
json.loads((data / 'quality.json').read_text(encoding='utf-8'))

In [ ]:
db.execute('SELECT COUNT(*), COUNT(DISTINCT line_id), COUNT(DISTINCT invoice_no), COUNT(DISTINCT stock_code), COUNT(DISTINCT country), MIN(invoice_ts), MAX(invoice_ts) FROM raw_lines').fetchall()

In [ ]:
db.execute("SELECT strftime(invoice_ts,'%Y-%m') AS month, COUNT(*) AS rows, SUM(customer_id IS NULL) AS unknown_customer_rows, ROUND(100.0*SUM(customer_id IS NULL)/COUNT(*),2) AS unknown_customer_pct FROM raw_lines GROUP BY 1 ORDER BY 1").fetchall()

In [ ]:
db.execute('SELECT SUM(n-1) AS extra_identical_rows FROM (SELECT COUNT(*) n FROM raw_lines GROUP BY invoice_no,stock_code,description,quantity,invoice_ts,unit_price,customer_id,country HAVING COUNT(*)>1)').fetchall()

In [ ]:
db.execute('SELECT COUNT(*) AS joined_rows, SUM(d.date_id IS NULL) AS missing_date_keys, SUM(c.country IS NULL) AS missing_country_keys, SUM(p.stock_code IS NULL) AS missing_product_keys FROM fact_retail_line f LEFT JOIN dim_date d ON f.date_id=d.date_id LEFT JOIN dim_country c ON f.country=c.country LEFT JOIN dim_product p ON f.stock_code=p.stock_code').fetchall()

In [ ]:
db.execute('SELECT COUNT(*) AS multi_country_or_customer_invoices FROM (SELECT invoice_no FROM raw_lines GROUP BY invoice_no HAVING COUNT(DISTINCT country)>1 OR COUNT(DISTINCT customer_id)>1)').fetchall()

In [ ]:
db.close()